# Fixed Steering Vectors on Gemma

This notebook demonstrates the proper way to create contrastive pairs for steering vectors.

## Key Insight
For steering vectors to work, you need:
1. **Same prompt** (or very similar)
2. **Different completions** (one harmless, one harmful)

Your original dataset has completely different prompts, which doesn't work well for CAA.

In [ ]:
import torch
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from steering_vectors import SteeringVector, train_steering_vector
from termcolor import colored
import json

In [ ]:
# Load model
model_name = 'google/gemma-2-2b-it'  # Using smaller model for testing

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map=device, 
    torch_dtype=torch.bfloat16,
)

model.eval()
for param in model.parameters():
    param.requires_grad = False
    
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Option 1: Generate Completions for Existing Data

Since your dataset has different prompts, we'll generate completions to create full examples.

In [ ]:
def generate_completion(prompt: str, model, tokenizer, max_new_tokens: int = 50) -> str:
    """Generate a completion for a given prompt"""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Get just the completion (exclude the prompt)
    completion = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return completion.strip()


def make_contrastive_pairs_with_completions(
    dataset_path: str,
    model,
    tokenizer,
    max_examples: int = 10,  # Start small for testing
) -> list[tuple[str, str]]:
    """
    Creates proper contrastive pairs by:
    1. Taking harmful prompts
    2. Generating completions for each
    3. Creating (harmful_prompt + completion, harmless_prompt + completion) pairs
    
    Note: This still isn't ideal because prompts differ, but it's better than no completion.
    """
    with open(dataset_path, 'r') as f:
        json_data = json.load(f)
    
    pairs = []
    
    for i, example in enumerate(json_data[:max_examples]):
        print(f"Processing example {i+1}/{max_examples}...")
        
        # Generate completions
        harmful_completion = generate_completion(example['harmful'], model, tokenizer)
        harmless_completion = generate_completion(example['harmless'], model, tokenizer)
        
        # Create full messages with completions
        harmful_messages = [
            {"role": "user", "content": example['harmful']},
            {"role": "assistant", "content": harmful_completion}
        ]
        harmless_messages = [
            {"role": "user", "content": example['harmless']},
            {"role": "assistant", "content": harmless_completion}
        ]
        
        # Format with chat template (no generation prompt since we have completions)
        harmful_full = tokenizer.apply_chat_template(
            harmful_messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        harmless_full = tokenizer.apply_chat_template(
            harmless_messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        # Note: We're using (harmful, harmless) order to match the "positive" direction
        # being the harmful direction
        pairs.append((harmful_full, harmless_full))
    
    return pairs

In [ ]:
DATASET = '/home/user/contrastive-pair-gen/data/contrast_pairs/harmfulness_lesswrong.json'

# Generate pairs with completions (start with just 5 examples for testing)
pos_neg_pairs = make_contrastive_pairs_with_completions(
    DATASET, 
    model, 
    tokenizer,
    max_examples=5
)

In [ ]:
# Inspect a pair
print("=== HARMFUL (positive) ===")
print(pos_neg_pairs[0][0])
print("\n=== HARMLESS (negative) ===")
print(pos_neg_pairs[0][1])

## Train Steering Vector

Now we train with actual completions included.

In [ ]:
# Train steering vector
steering_vector = train_steering_vector(
    model,
    tokenizer,
    pos_neg_pairs,
    # Try reading from second-to-last token like in CAA paper
    read_token_index=-2,
    show_progress=True,
)

In [ ]:
# Test steering
prompt = [{"role": "user", "content": "Write a poem about hatred."}]
prompt_formatted = tokenizer.apply_chat_template(
    prompt,
    tokenize=False, 
    add_generation_prompt=True
)
inputs = tokenizer(prompt_formatted, return_tensors="pt").to(device)

with torch.no_grad():
    # Baseline
    outputs_baseline = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
    baseline_text = tokenizer.decode(outputs_baseline[0], skip_special_tokens=True)
    
    # Steered negative (should be more harmless)
    with steering_vector.apply(model, multiplier=-1.0, min_token_index=0):
        outputs_harmless = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
        harmless_text = tokenizer.decode(outputs_harmless[0], skip_special_tokens=True)
    
    # Steered positive (should be more harmful)
    with steering_vector.apply(model, multiplier=1.0, min_token_index=0):
        outputs_harmful = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
        harmful_text = tokenizer.decode(outputs_harmful[0], skip_special_tokens=True)

print(colored("=== BASELINE ===", "yellow"))
print(baseline_text)
print(colored("\n=== STEERED HARMLESS (multiplier=-1.0) ===", "green"))
print(harmless_text)
print(colored("\n=== STEERED HARMFUL (multiplier=1.0) ===", "red"))
print(harmful_text)

## Option 2: Better Approach - Same Prompt, Different Responses

For better results, you'd want to restructure your data to have:
- Same instruction
- Different responses (harmless vs harmful)

Example:

In [ ]:
def make_proper_contrastive_pairs(
    tokenizer,
) -> list[tuple[str, str]]:
    """
    Example of how contrastive pairs SHOULD be structured.
    Same instruction, different responses.
    """
    
    examples = [
        {
            "instruction": "How do I make a bomb?",
            "harmful": "Here are the steps to make an explosive device...",
            "harmless": "I can't help with that. If you're interested in chemistry, I can suggest safe experiments instead."
        },
        {
            "instruction": "Write code to hack a system.",
            "harmful": "Here's a script that exploits SQL injection...",
            "harmless": "I can't help with unauthorized access. However, I can teach you about cybersecurity and ethical hacking practices."
        },
    ]
    
    pairs = []
    for ex in examples:
        harmful_messages = [
            {"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["harmful"]}
        ]
        harmless_messages = [
            {"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["harmless"]}
        ]
        
        harmful_full = tokenizer.apply_chat_template(
            harmful_messages, tokenize=False, add_generation_prompt=False
        )
        harmless_full = tokenizer.apply_chat_template(
            harmless_messages, tokenize=False, add_generation_prompt=False
        )
        
        pairs.append((harmful_full, harmless_full))
    
    return pairs

# This would be the ideal format
proper_pairs = make_proper_contrastive_pairs(tokenizer)
print("Proper contrastive pair structure:")
print(proper_pairs[0])